LAYOFF SEVERITY SEGMENTATION
___
BUSINESS QUESTION

1. Can we group each layoff event into a clear, business-readable severity tier (Mild / Moderate / Severe / Critical) based on the PERCENTAGE of staff laid off?
2. Once we have these tiers, do certain industries, funding brackets, or repeat-layoff companies show up more often in the more severe tiers?

In [1]:
import pandas as pd
import numpy as np

Load the master dataset from Analysis 2

In [2]:
df = pd.read_csv("D:\DATA ANALYSIS\MySQL data project(A)\Data\Processed\layoffs_master_enriched.csv", parse_dates=["date"])

Define severity tiers using pd.cut()

In [3]:
severity_bins = [-0.01, 0.10, 0.25, 0.50, 1.01]

severity_labels = ["Mild (<=10%)", "Moderate (10-25%)", "Severe (25-50%)", "Critical (>50%)"]

df["severity_tier"] = pd.cut(
    df["percentage_laid_off"], bins=severity_bins, labels=severity_labels
)

Check how many events fall into each tier

In [4]:
print("Layoff events per severity tier:")
print(df["severity_tier"].value_counts().sort_index())

Layoff events per severity tier:
severity_tier
Mild (<=10%)         496
Moderate (10-25%)    610
Severe (25-50%)      301
Critical (>50%)      164
Name: count, dtype: int64


Cross-tabulate severity tier against industry

In [5]:
industry_severity_table = pd.pivot_table(
    df,
    index="industry",
    columns="severity_tier",
    values="company",   # any column works here since we're just counting rows
    aggfunc="size",
    fill_value=0,        # if a combination never occurs, show 0 instead of blank
    observed=True,
)

print("\nIndustry x Severity Tier (event counts):")
print(industry_severity_table)


Industry x Severity Tier (event counts):
severity_tier   Mild (<=10%)  Moderate (10-25%)  Severe (25-50%)  \
industry                                                           
Aerospace                  0                  2                0   
Construction               4                  3                3   
Consumer                  22                 28               13   
Crypto                    16                 33               24   
Data                      22                 14               12   
Education                 17                 15                9   
Energy                     1                  5                1   
Fin-Tech                   0                  2                1   
Finance                   57                 87               37   
Fitness                    2                 12                4   
Food                      31                 23               13   
HR                        12                 17               11   
Hardwa

Check whether repeat-layoff companies skew toward higher severity

In [6]:
repeat_vs_severity = pd.crosstab(
    df["is_repeat_layoff_company"], df["severity_tier"], normalize="index"
)
repeat_vs_severity = (repeat_vs_severity * 100).round(1)


print("\n% of events in each severity tier, by repeat-layoff status:")
print(repeat_vs_severity)


% of events in each severity tier, by repeat-layoff status:
severity_tier             Mild (<=10%)  Moderate (10-25%)  Severe (25-50%)  \
is_repeat_layoff_company                                                     
False                             29.8               37.3             19.8   
True                              35.4               42.2             17.7   

severity_tier             Critical (>50%)  
is_repeat_layoff_company                   
False                                13.1  
True                                  4.7  


Save the enriched master dataset

In [7]:
df.to_csv("layoffs_master_enriched.csv", index=False)

print("\nUpdated: layoffs_master_enriched.csv")
print(f"Columns now in the master dataset: {list(df.columns)}")


Updated: layoffs_master_enriched.csv
Columns now in the master dataset: ['company', 'location', 'industry', 'total_laid_off', 'percentage_laid_off', 'date', 'stage', 'country', 'funds_raised_millions', 'number_of_layoff_events', 'is_repeat_layoff_company', 'log_funds_raised', 'funding_bracket', 'severity_tier']
